# Deep Learning NLP Text Classifier — SMS Spam Detection

Build a neural network that classifies SMS messages as **spam** or **ham**.

This notebook demonstrates Keras TextVectorization, an Embedding layer, multiple Dense layers, Batch Normalization, Dropout, Early Stopping, training curves, test evaluation, sample inference confidences, and saving a trained `.keras` model.

**Dataset:** SMS Spam Collection (public TSV mirror). The first run requires internet access.

**Leakage prevention:** split the dataset first; adapt the text vocabulary on training text only.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

## 1. Load and inspect the SMS dataset

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\\t", header=None, names=["label", "text"])
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["target"] = df["label"].map({"ham": 0, "spam": 1})

print("Dataset shape:", df.shape)
display(df["label"].value_counts().to_frame("count"))
display(df.head())

## 2. Split into training, validation, and test sets
The test set is held back for final evaluation. The vectorizer vocabulary will be learned from training messages only.

In [ ]:
X = df["text"].astype(str).to_numpy()
y = df["target"].to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))

## 3. Tokenize and vectorize text
`TextVectorization` converts raw messages into integer token sequences. `adapt()` is called on training text only.

In [ ]:
MAX_TOKENS = 10000
SEQUENCE_LENGTH = 100

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(64))

vocabulary_size = len(vectorizer.get_vocabulary())
print("Vocabulary size:", vocabulary_size)
print("Example message:", X_train[0])
print("Token IDs:", vectorizer(tf.constant([X_train[0]])).numpy()[0][:20])

## 4. Build the deep learning model
The network uses an Embedding layer, two Dense layers, Batch Normalization, Dropout, and a sigmoid output for spam probability.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(vocabulary_size, 64, mask_zero=True),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.40),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
model.summary()

## 5. Train with Early Stopping

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Plot training and validation loss/accuracy

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(history_df["loss"], label="Training loss")
axes[0].plot(history_df["val_loss"], label="Validation loss")
axes[0].set_title("Loss convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_df["accuracy"], label="Training accuracy")
axes[1].plot(history_df["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Accuracy convergence")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Evaluate on the unseen test set

In [ ]:
test_metrics = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("Test metrics:")
for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}")

test_probabilities = model.predict(X_test, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)

print(classification_report(
    y_test, test_predictions, target_names=["ham", "spam"], zero_division=0
))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_probabilities), 4))

## 8. Confusion matrix and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=["ham", "spam"], cmap="Blues", ax=axes[0]
)
axes[0].set_title("Test confusion matrix")

fpr, tpr, _ = roc_curve(y_test, test_probabilities)
auc_value = roc_auc_score(y_test, test_probabilities)
axes[1].plot(fpr, tpr, label=f"Neural Network (AUC={auc_value:.3f})")
axes[1].plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Test ROC curve")
axes[1].legend()

plt.tight_layout()
plt.savefig("test_evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Test on unseen sample messages
Confidence is the probability assigned to the predicted class; it is not a guarantee of correctness.

In [ ]:
sample_messages = [
    "Congratulations! You have won a free prize. Call now to claim.",
    "Hey, are we still meeting for lunch today?",
    "URGENT! Your account has been selected for a cash reward. Reply WIN."
]

spam_probabilities = model.predict(np.array(sample_messages), verbose=0).ravel()
sample_rows = []
for message, spam_probability in zip(sample_messages, spam_probabilities):
    predicted = "spam" if spam_probability >= 0.5 else "ham"
    confidence = spam_probability if predicted == "spam" else 1 - spam_probability
    sample_rows.append({
        "message": message,
        "predicted_label": predicted,
        "spam_probability": round(float(spam_probability), 4),
        "prediction_confidence": round(float(confidence), 4)
    })

sample_results = pd.DataFrame(sample_rows)
display(sample_results)
sample_results.to_csv("sample_inference_results.csv", index=False)

## 10. Save the trained model and training history

In [ ]:
MODEL_PATH = "sms_spam_text_classifier.keras"
model.save(MODEL_PATH)
history_df.to_csv("training_history.csv", index=False)

print("Saved:", MODEL_PATH)
print("Saved: training_curves.png")
print("Saved: test_evaluation_plots.png")
print("Saved: sample_inference_results.csv")
print("Saved: training_history.csv")

## 11. Reload and verify the saved model

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)
new_message = "Free entry in our weekly competition. Text WIN to 80000."
probability = float(loaded_model.predict(np.array([new_message]), verbose=0)[0][0])
label = "spam" if probability >= 0.5 else "ham"

print("Message:", new_message)
print("Predicted class:", label)
print("Spam probability:", round(probability, 4))

## 12. Submission checklist

- [x] Text tokenization/vectorization with Keras `TextVectorization`
- [x] Embedding-based neural network with multiple Dense layers
- [x] Batch Normalization and Dropout
- [x] Early Stopping callback
- [x] Training/validation loss and accuracy curves
- [x] Evaluation on unseen test data
- [x] Confusion matrix and ROC-AUC
- [x] Sample inference with confidence values
- [x] Save/reload the trained model

### Generated files (after running all cells)
- `sms_spam_text_classifier.keras`
- `training_curves.png`
- `test_evaluation_plots.png`
- `sample_inference_results.csv`
- `training_history.csv`

Run all cells before submission. Upload the notebook and generated outputs to GitHub.